# RT moment reconstruction: normal conformity and continuous-test balance

This notebook uses the canonical face/volume construction of [Barrenechea et al. (2026)](https://doi.org/10.1137/24M1673073), equations (4.9) and (5.1). RT0, RT1 and RT2 are supported. Normal data on macro boundaries come from the MHM skeleton; interior normal moments average the two raw one-sided fluxes; interior vector moments preserve the raw flux.

The analytical examples below are pyMHM calculations. They do not reproduce digitized paper curves or execute the paper's FreeFem++ solver.


The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

import os
HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from pymhm import FaceSpace, SkeletonSpace, TriangleMesh
from pymhm.recovery.moments import reconstruct_darcy_moments


## A primal P3 pressure and RT2 reconstruction

Take `p=sin(2πx)sin(2πy)`, `K=I`, homogeneous Dirichlet pressure and `f=8π²p`. The skeletal degree is one and each macrotriangle contains four fine triangles. The primal local degree three satisfies the two-dimensional hypothesis `k >= ell+2` for the paper's optimal flux estimates.


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy


def pressure(points):
    """Analytical sine pressure on the unit square."""
    return np.sin(2 * np.pi * points[:, 0]) * np.sin(2 * np.pi * points[:, 1])


def source(points):
    """Negative Laplacian of the pressure."""
    return 8 * np.pi**2 * pressure(points)


def exact_flux(points):
    """Physical flux minus the analytical pressure gradient."""
    x, y = 2 * np.pi * points.T
    return -2 * np.pi * np.column_stack((np.cos(x) * np.sin(y), np.sin(x) * np.cos(y)))


mesh = TriangleMesh.unit_square(2)
skeleton = SkeletonSpace(mesh, tuple((FaceSpace.uniform(1) for _ in mesh.faces)))
solution_definition = define_darcy(
    mesh, source=source, skeleton=skeleton, degree=3, local_refinement=2, quadrature_order=8
)
solution_system = assemble(solution_definition.problem)
solution = recover_darcy(
    solution_definition,
    solution_system,
    solution_system.solve(constraints=pressure_constraints(solution_definition, solution_system)),
)
recovered = reconstruct_darcy_moments(solution, degree=2)
{
    "raw_flux_L2": solution.flux_l2_error(exact_flux, 10),
    "RT2_flux_L2": recovered.flux_l2_error(exact_flux, 10),
}


## Conservation has a precise test space

The identity is `(div(σ)-f,v)_K=0` for **continuous** piecewise P2 tests on each macrocell. It does not imply a separate balance against every discontinuous fine-cell constant. The following assertions check both the vanishing continuous moments and a genuinely nonzero fine-cell defect; neither result is hidden by a projection.


In [ ]:
continuous_defect = max(float(np.max(np.abs(v))) for v in recovered.continuous_moment_residuals())
fine_defect = max(float(np.max(np.abs(v))) for v in recovered.fine_conservation_residuals())
normal_defect = max(float(np.max(np.abs(v))) for v in recovered.normal_flux_residuals())
assert continuous_defect < 2e-12
assert normal_defect < 1e-14
assert fine_defect > 1e-5
assert np.max(np.abs(recovered.conservation_residuals())) < 2e-12
{"continuous_P2_moment_max": continuous_defect,
 "fine_DG0_integral_max": fine_defect,
 "normal_moment_max": normal_defect,
 "raw_RT2_divergence_L2": recovered.divergence_l2_error(source, 10),
 "projected_divergence_L2": recovered.projected_divergence_l2_error(source, 10)}


## Five-mesh analytical study

The archived study uses macro resolutions 1, 2, 4, 8, 16; local refinement two; RT2 recovery; and `(ell,k)=(0,2)` or `(1,3)`. It compares actual raw and reconstructed flux errors, raw divergence errors, and **continuous projected divergence** errors as separate quantities. Reconstruction does not guarantee a smaller flux L2 error on each finite mesh.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    records = json.loads((ROOT / "examples/results/reconstruction-moments.json").read_text())
    [row for row in records["convergence"] if row["macro_resolution"] == 16]


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    for name in ("flux-fields", "divergence-fields", "convergence-and-conservation"):
        display(Image(filename=str(ROOT / f"docs/figures/reconstruction-moments/{name}.png")))


## Verification and limitations

Native DOLFINx/Basix tests independently compare canonical RT interpolation and the UFL divergence for mathematical RT orders 0, 1, 2 (Basix degrees 1, 2, 3). Portable tests verify face and volume moments, Piola orientation, within-macro and cross-macro normal continuity, and the divergence commuting identity.

For a material jump on an interior fine face, pass explicit one-sided `raw_fluxes`, whose evaluators receive physical points and fine-cell indices. The default permeability callback cannot infer two different material limits at one point. The supported geometry is affine triangles, with skeletal partitions aligned to fine boundary vertices.

The moment reconstruction is distinct from an energy-minimizing RT0 correction. By itself it is also not the paper's complete a posteriori estimator, which additionally needs potential reconstruction, divergence correction and oscillation terms.
